# 1.15 同一模型為什麼會寫出不同答案？

# 第 1 章：從猜下一個字開始

前置：讀過入門指南的 Python 與 tensor 部分即可。這一章只用幾個字；不要先開完整 Transformer。每段 code 都可單獨執行。




In [ ]:
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch
from torch import nn
from torch.nn import functional as F

torch.set_num_threads(1)
torch.manual_seed(42)

**先看圖：把分數換成比例：像分一塊餅**

一起加同常數不改比例，分數差才重要。

橘色邊框只提示閱讀順序，靜態標註一直保留。系統的減少動態效果設定會停用動畫。


In [ ]:
from IPython.display import SVG, display

display(SVG(filename=str(root / "course/figures/softmax.svg")))

**先想一想：**增加溫度會讓最大候選更佔優勢嗎？

greedy 永遠取最大機率；sampling 保留其他候選。temperature 改變分佈尖銳程度，沒有更改權重，因此不是性格訓練。

**把直覺寫成數學：**$p_i(T)=softmax(z/T)_i$，$T>0$；$T$ 小接近 greedy，大則較平坦。

**最小實驗：**以下程式只處理這節的問題。


In [ ]:
z = torch.tensor([3.0, 1.0, 0.0])
for temperature in (0.5, 1.0, 2.0):
    print(temperature, (z / temperature).softmax(0))
print("greedy", z.argmax().item())

**如何讀結果：**比較行為微調時固定抽樣設定，避免把抽樣變化當成訓練效果。

**只改一件事：**只把最後一個分數提高，比較三個溫度。
